**Create Dataset**

In [4]:
TRAINING_DATA_PATH = "training_data/synth_train_data.npz"

In [3]:
import librosa
import torch
import numpy as np
from helper_functions import AudioRecordingDataset, generate_dataset

In [5]:
X, Y, HV = generate_dataset()
np.savez(file = TRAINING_DATA_PATH, X = X,Y = Y, hv = HV)

/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=173
  warnings.warn(


**Load Torch Dataset/Dataloader**

In [18]:
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

data = AudioRecordingDataset(TRAINING_DATA_PATH)
val_mask = (data.hv == 0.25)
val_idx = np.where(val_mask)[0]
train_idx = np.where(~val_mask)[0]


train_X = data.X[train_idx]
data.mean = train_X.mean(axis = 0)
data.std = train_X.std(axis = 0) + 1e-8 #for 0 values

print(f"data_mean: {data.mean.shape}| data_std: {data.std.shape}")

train_data = Subset(data, train_idx)
val_data = Subset(data, val_idx)

print(f"train {len(train_data)} | val {len(val_data)}")

train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
val_loader = DataLoader(
    dataset = val_data,
    batch_size = 32,
    shuffle = True
)

# batch size x feature_size (64, 84)
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

data_mean: (84,)| data_std: (84,)
train 1760 | val 440


In [19]:
num_epochs = 35

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(val_loader):.3f}| acc {correct/total:.3f}")


Epoch   1 | train 4.198 | val 3.708| acc 0.402
Epoch   2 | train 3.270 | val 2.685| acc 0.636
Epoch   3 | train 2.351 | val 1.870| acc 0.820
Epoch   4 | train 1.713 | val 1.356| acc 0.911
Epoch   5 | train 1.322 | val 1.039| acc 0.936
Epoch   6 | train 1.051 | val 0.828| acc 0.968
Epoch   7 | train 0.859 | val 0.672| acc 0.975
Epoch   8 | train 0.729 | val 0.562| acc 0.982
Epoch   9 | train 0.610 | val 0.474| acc 0.989
Epoch  10 | train 0.537 | val 0.408| acc 0.991
Epoch  11 | train 0.481 | val 0.355| acc 0.993
Epoch  12 | train 0.417 | val 0.312| acc 0.995
Epoch  13 | train 0.375 | val 0.277| acc 1.000
Epoch  14 | train 0.340 | val 0.248| acc 0.993
Epoch  15 | train 0.297 | val 0.221| acc 1.000
Epoch  16 | train 0.275 | val 0.201| acc 1.000
Epoch  17 | train 0.251 | val 0.181| acc 1.000
Epoch  18 | train 0.236 | val 0.167| acc 0.993
Epoch  19 | train 0.209 | val 0.152| acc 0.998
Epoch  20 | train 0.198 | val 0.140| acc 1.000
Epoch  21 | train 0.183 | val 0.129| acc 1.000
Epoch  22 | t

In [20]:
# Save the Model
checkpoint = {
    'model_state': model.state_dict(),
    'mean' : torch.tensor(data.mean, dtype=torch.float32),
    'std' : torch.tensor(data.std, dtype=torch.float32)
}

torch.save(checkpoint, 'models/linear_synth.pth')

**Test Validation**

Load Model

In [21]:
VALIDATION_DATA_PATH = "nsynth-test"
VALIDATION_PROCESSED_DATA = "validation_data"


In [23]:
#Generate Validate dataset
from load_validation import load_data

A,B = load_data()

np.savez(f"{VALIDATION_PROCESSED_DATA}/{VALIDATION_DATA_PATH}", X=A, Y=B)


/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=136
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=128
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=164
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=224
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=112
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=56
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/li

In [24]:
# Validation Data on Linear Classifification Trained on Synthetic Data

checkpoint = torch.load('models/linear_synth.pth')

# Define model Architecture
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)

d =  np.load(f"{VALIDATION_PROCESSED_DATA}/{VALIDATION_DATA_PATH}.npz")
X = torch.tensor(d['X'], dtype  = torch.float32)
Y = torch.tensor(d['Y'], dtype = torch.long)

#norm
X= (X - checkpoint['mean']) / checkpoint['std']

model.load_state_dict(checkpoint['model_state'])
model.eval()

with torch.no_grad():
    logits = model(X)
acc = (logits.argmax(1) == Y).float().mean().item()
print(f"val acc {acc:.3f}")



val acc 0.698


TODO: Add commenting to all refactored functions (params, returns, brief)